In [ ]:
# ERCOT Load Prediction Using Zone-Level Weather Data
# Goal: Predict the power grid load by ERCOT weather zone
# Models: Naive Baseline, Random Forest, LightGBM, LTSM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path

PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
ercot_files = [
    DATA_DIR / "Native_Load_2022.xlsx",
    DATA_DIR / "Native_Load_2023.xlsx",
    DATA_DIR / "Native_Load_2024.xlsx",
    DATA_DIR / "Native_Load_2025.xlsx",
    DATA_DIR / "Native_Load_2026.xlsx",
]

ercot_dfs = []
for file in ercot_files:
    temp = pd.read_excel(file)
    temp["source_file"] = file.name
    ercot_dfs.append(temp)

df_load = pd.concat(ercot_dfs, ignore_index=True)
df_load.head()

In [ ]:
df_load.columns = [c.strip() for c in df_load.columns]

zone_cols = ["COAST", "EAST", "FWEST", "NORTH", "NCENT", "SOUTH", "SCENT", "WEST", "ERCOT"]

df_load["Hour Ending"] = pd.to_datetime(df_load["Hour Ending"], errors="coerce")
df_load = df_load.rename(columns={"Hour Ending": "datetime"})

for col in zone_cols:
    df_load[col] = pd.to_numeric(df_load[col], errors="coerce")

df_load = df_load.sort_values("datetime").drop_duplicates(subset=["datetime"]).reset_index(drop=True)

print(df_load.shape)
df_load.head()

In [ ]:
weather_file = DATA_DIR / "open-meteo.csv"
locations_file = DATA_DIR / "open-meteo-locations.csv"

df_weather = pd.read_csv(weather_file)
df_weather_locations = pd.read_csv(locations_file)

print("weather shape:", df_weather.shape)
print("locations shape:", df_weather_locations.shape)

display(df_weather.head(3))
display(df_weather_locations.head(10))

In [ ]:
display(df_weather_locations)

In [ ]:
location_to_zone = {
    0: "COAST",   # Houston
    1: "EAST",    # Tyler
    2: "FWEST",   # Midland
    3: "NCENT",   # Dallas / DFW
    4: "SOUTH",   # Corpus Christi
    5: "SCENT",   # Austin / San Antonio
    6: "WEST",    # Abilene
    7: "NORTH"    # Wichitia Falls
}

In [ ]:
df_weather.columns = [c.strip() for c in df_weather.columns]

rename_weather = {
    "temperature_2m (°C)": "temp",
    "precipitation (mm)": "precip",
    "relative_humidity_2m (%)": "humidity",
    "apparent_temperature (°C)": "apparent_temp",
    "wind_speed_10m (km/h)": "wind_speed",
    "shortwave_radiation (W/m²)": "shortwave_radiation",
}
df_weather = df_weather.rename(columns=rename_weather)

df_weather["datetime"] = pd.to_datetime(df_weather["time"], unit="s", utc=False, errors="coerce")
df_weather["zone"] = df_weather["location_id"].map(location_to_zone)

weather_keep = [
    "location_id",
    "zone",
    "datetime",
    "temp",
    "precip",
    "humidity",
    "apparent_temp",
    "wind_speed",
    "shortwave_radiation",
]

df_weather = df_weather[weather_keep].copy()
df_weather = df_weather.dropna(subset=["datetime", "zone"]).sort_values(["zone", "datetime"])

print(df_weather.shape)
df_weather.head()

In [ ]:
print(df_weather["zone"].value_counts(dropna=False))
print(df_weather["datetime"].min(), "to", df_weather["datetime"].max())
display(df_weather.describe(include="all"))

In [ ]:
weather_vars = ["temp", "precip", "humidity", "apparent_temp", "wind_speed", "shortwave_radiation"]

df_weather_wide = df_weather.pivot(index="datetime", columns="zone", values=weather_vars)

df_weather_wide.columns = [f"{var}_{zone}" for var, zone in df_weather_wide.columns]
df_weather_wide = df_weather_wide.reset_index().sort_values("datetime")

print(df_weather_wide.shape)
df_weather_wide.head()

In [ ]:
df = df_load.merge(df_weather_wide, on="datetime", how="left")

print(df.shape)
df.head()

In [ ]:
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["dayofyear"] = df["datetime"].dt.dayofyear
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df.head()

In [ ]:
df["ERCOT_current"] = df["ERCOT"]
df["target_24h"] = df["ERCOT"].shift(-24)
target = "target_24h"

for lag in [1, 2, 3, 24, 48, 168]:
    df[f"ERCOT_lag_{lag}"] = df["ERCOT"].shift(lag)

df["ERCOT_roll_mean_24"] = df["ERCOT"].shift(1).rolling(24).mean()
df["ERCOT_roll_std_24"] = df["ERCOT"].shift(1).rolling(24).std()
df["ERCOT_roll_mean_168"] = df["ERCOT"].shift(1).rolling(168).mean()
df["ERCOT_roll_std_168"] = df["ERCOT"].shift(1).rolling(168).std()

df = df.dropna()

In [ ]:
temp_cols = [c for c in df.columns if c.startswith("temp_")]
apparent_cols = [c for c in df.columns if c.startswith("apparent_temp_")]

for col in temp_cols:
    zone = col.replace("temp_", "")
    df[f"cdd_{zone}"] = np.maximum(df[col] - 18.0, 0)   # Celsius base ~18C
    df[f"hdd_{zone}"] = np.maximum(18.0 - df[col], 0)

for col in apparent_cols:
    zone = col.replace("apparent_temp_", "")
    df[f"cdd_app_{zone}"] = np.maximum(df[col] - 18.0, 0)
    df[f"hdd_app_{zone}"] = np.maximum(18.0 - df[col], 0)

df.head()

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
display(missing[missing > 0].head(50))

In [ ]:
drop_cols = [
    # "COAST", "EAST", "FWEST", "NORTH",
    # "NCENT", "SOUTH", "SCENT", "WEST"
]

df = df.drop(columns=drop_cols)

drop_cols = ["datetime", "source_file"]
feature_cols = [c for c in df.columns if c not in drop_cols + [target]]

model_df = df.dropna(subset=feature_cols + [target]).copy()

print("model_df shape:", model_df.shape)
model_df.head()

In [ ]:
train_df = model_df[model_df["datetime"] < "2024-01-01"].copy()
val_df   = model_df[(model_df["datetime"] >= "2024-01-01") & (model_df["datetime"] < "2025-01-01")].copy()
test_df  = model_df[(model_df["datetime"] >= "2025-01-01") & (model_df["datetime"] < "2026-01-01")].copy()
test_2026_df = model_df[model_df["datetime"] >= "2026-01-01"].copy()

X_train = train_df[feature_cols]
y_train = train_df[target]

X_val = val_df[feature_cols]
y_val = val_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

X_test_2026 = test_2026_df[feature_cols]
y_test_2026 = test_2026_df[target]

print("Train:", train_df["datetime"].min(), "to", train_df["datetime"].max(), train_df.shape)
print("Val  :", val_df["datetime"].min(), "to", val_df["datetime"].max(), val_df.shape)
print("Test :", test_df["datetime"].min(), "to", test_df["datetime"].max(), test_df.shape)
print("2026 :", test_2026_df["datetime"].min(), "to", test_2026_df["datetime"].max(), test_2026_df.shape)

In [ ]:
import random
import json
from pathlib import Path
from sklearn.model_selection import ParameterGrid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = OUTPUT_DIR

# For cleaner forecasting features, exclude zone-load decomposition
zone_load_cols = ["COAST", "EAST", "FWEST", "NORTH", "NCENT", "SOUTH", "SCENT", "WEST"]

model_df = model_df.copy()

# Aggregate weather features for both tabular models and LSTM
temp_cols = [c for c in model_df.columns if c.startswith("temp_")]
app_temp_cols = [c for c in model_df.columns if c.startswith("apparent_temp_")]
wind_cols = [c for c in model_df.columns if c.startswith("wind_speed_")]
rad_cols = [c for c in model_df.columns if c.startswith("shortwave_radiation_")]
humid_cols = [c for c in model_df.columns if c.startswith("humidity_")]
precip_cols = [c for c in model_df.columns if c.startswith("precip_")]

if temp_cols:
    model_df["temp_mean"] = model_df[temp_cols].mean(axis=1)
if app_temp_cols:
    model_df["apparent_temp_mean"] = model_df[app_temp_cols].mean(axis=1)
if wind_cols:
    model_df["wind_mean"] = model_df[wind_cols].mean(axis=1)
if rad_cols:
    model_df["radiation_mean"] = model_df[rad_cols].mean(axis=1)
if humid_cols:
    model_df["humidity_mean"] = model_df[humid_cols].mean(axis=1)
if precip_cols:
    model_df["precip_mean"] = model_df[precip_cols].mean(axis=1)

if "temp_mean" in model_df.columns:
    model_df["cdd_mean"] = np.maximum(model_df["temp_mean"] - 18.0, 0)
    model_df["hdd_mean"] = np.maximum(18.0 - model_df["temp_mean"], 0)

exclude_cols = ["datetime", "source_file", target] + zone_load_cols
feature_cols = [c for c in model_df.columns if c not in exclude_cols]

print(f"Number of tabular features: {len(feature_cols)}")
print(feature_cols[:25])

In [ ]:
# Shared split definition for all tabular models
train_df = model_df[model_df["datetime"] < "2024-01-01"].copy()
val_df = model_df[(model_df["datetime"] >= "2024-01-01") & (model_df["datetime"] < "2025-01-01")].copy()
test_df = model_df[(model_df["datetime"] >= "2025-01-01") & (model_df["datetime"] < "2026-01-01")].copy()
test_2026_df = model_df[model_df["datetime"] >= "2026-01-01"].copy()

X_train = train_df[feature_cols]
y_train = train_df[target]

X_val = val_df[feature_cols]
y_val = val_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

X_test_2026 = test_2026_df[feature_cols]
y_test_2026 = test_2026_df[target]

split_summary = pd.DataFrame([
    {"Split": "Train", "Start": train_df["datetime"].min(), "End": train_df["datetime"].max(), "Rows": len(train_df)},
    {"Split": "Validation_2024", "Start": val_df["datetime"].min(), "End": val_df["datetime"].max(), "Rows": len(val_df)},
    {"Split": "Test_2025", "Start": test_df["datetime"].min(), "End": test_df["datetime"].max(), "Rows": len(test_df)},
    {"Split": "Test_2026_YTD", "Start": test_2026_df["datetime"].min(), "End": test_2026_df["datetime"].max(), "Rows": len(test_2026_df)},
])

display(split_summary)

In [ ]:
def config_to_str(config):
    if isinstance(config, str):
        return config
    return json.dumps(config, sort_keys=True)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None))) * 100
    r_squared = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R_squared": r_squared,
    }

search_results = []
final_results = []
prediction_frames = {}

def record_metrics(container, model_name, config, split_name, y_true, y_pred):
    row = {
        "Model": model_name,
        "Config": config_to_str(config),
        "Split": split_name,
        **compute_metrics(y_true, y_pred),
    }
    container.append(row.copy())
    return row

def store_predictions(model_name, split_name, datetimes, y_true, y_pred):
    prediction_frames[(model_name, split_name)] = pd.DataFrame({
        "datetime": pd.to_datetime(datetimes),
        "actual": np.asarray(y_true, dtype=float),
        "pred": np.asarray(y_pred, dtype=float),
    })

def print_row(row):
    print(
        f'{row["Model"]} | {row["Split"]} | '
        f'MAE={row["MAE"]:,.3f} | RMSE={row["RMSE"]:,.3f} | '
        f'MAPE={row["MAPE"]:,.3f}% | R_squared={row["R_squared"]:.5f}'
    )

In [ ]:
# Naive 24-hour-ahead baseline: tomorrow same as now
baseline_config = {"rule": "predict target_24h using ERCOT_current"}

for split_name, split_df, y_split in [
    ("Validation_2024", val_df, y_val),
    ("Test_2025", test_df, y_test),
    ("Test_2026_YTD", test_2026_df, y_test_2026),
]:
    baseline_pred = split_df["ERCOT_current"].values
    row = record_metrics(final_results, "Baseline_CurrentERCOT", baseline_config, split_name, y_split, baseline_pred)
    print_row(row)
    store_predictions("Baseline_CurrentERCOT", split_name, split_df["datetime"], y_split, baseline_pred)

In [ ]:
# Random Forest validation search
rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [12, 18],
    "min_samples_leaf": [1, 2],
}

best_rf_model = None
best_rf_params = None
best_rf_score = -np.inf

for params in ParameterGrid(rf_param_grid):
    rf_model = RandomForestRegressor(
        **params,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)

    val_pred = rf_model.predict(X_val)
    row = record_metrics(search_results, "RandomForest", params, "Validation_2024", y_val, val_pred)

    if row["R_squared"] > best_rf_score:
        best_rf_score = row["R_squared"]
        best_rf_params = params
        best_rf_model = rf_model

rf_search_df = (
    pd.DataFrame([r for r in search_results if r["Model"] == "RandomForest"])
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

display(rf_search_df.head(10))
print("Best RF params:", best_rf_params)
print("Best RF Validation R_squared:", best_rf_score)

In [ ]:
# Best Random Forest: validation + 2025 test + 2026 YTD
for split_name, X_split, y_split, dt_split in [
    ("Validation_2024", X_val, y_val, val_df["datetime"]),
    ("Test_2025", X_test, y_test, test_df["datetime"]),
    ("Test_2026_YTD", X_test_2026, y_test_2026, test_2026_df["datetime"]),
]:
    rf_pred = best_rf_model.predict(X_split)
    row = record_metrics(final_results, "RandomForest", best_rf_params, split_name, y_split, rf_pred)
    print_row(row)
    store_predictions("RandomForest", split_name, dt_split, y_split, rf_pred)

rf_importances = pd.Series(best_rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

display(rf_importances.head(25))

plt.figure(figsize=(10, 8))
rf_importances.head(20).sort_values().plot(kind="barh")
plt.title("Random Forest Top 20 Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
import lightgbm as lgb

best_lgb_model = None
best_lgb_params = None
best_lgb_score = -np.inf

lgb_param_grid = {
    "n_estimators": [500, 1000],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [31, 63],
}

for params in ParameterGrid(lgb_param_grid):
    lgb_model = lgb.LGBMRegressor(
        objective="regression",
        random_state=SEED,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        min_child_samples=20,
        **params,
    )

    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    val_pred = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration_)
    row = record_metrics(search_results, "LightGBM", params, "Validation_2024", y_val, val_pred)

    if row["R_squared"] > best_lgb_score:
        best_lgb_score = row["R_squared"]
        best_lgb_params = params
        best_lgb_model = lgb_model

lgb_search_df = (
    pd.DataFrame([r for r in search_results if r["Model"] == "LightGBM"])
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

display(lgb_search_df.head(10))
print("Best LightGBM params:", best_lgb_params)
print("Best LightGBM Validation R_squared:", best_lgb_score)

In [ ]:
for split_name, X_split, y_split, dt_split in [
    ("Validation_2024", X_val, y_val, val_df["datetime"]),
    ("Test_2025", X_test, y_test, test_df["datetime"]),
    ("Test_2026_YTD", X_test_2026, y_test_2026, test_2026_df["datetime"]),
]:
    lgb_pred = best_lgb_model.predict(X_split, num_iteration=best_lgb_model.best_iteration_)
    row = record_metrics(final_results, "LightGBM", best_lgb_params, split_name, y_split, lgb_pred)
    print_row(row)
    store_predictions("LightGBM", split_name, dt_split, y_split, lgb_pred)

lgb_importances = pd.Series(best_lgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

display(lgb_importances.head(25))

plt.figure(figsize=(10, 8))
lgb_importances.head(20).sort_values().plot(kind="barh")
plt.title("LightGBM Top 20 Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
# LSTM feature set
lstm_features = [
    "ERCOT_current",
    "ERCOT_lag_24",
    "ERCOT_lag_168",
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "is_weekend",
    "temp_mean",
    "apparent_temp_mean",
    "humidity_mean",
    "wind_mean",
    "radiation_mean",
    "cdd_mean",
    "hdd_mean",
]

lstm_features = [c for c in lstm_features if c in model_df.columns]

lstm_df = model_df[["datetime", target] + lstm_features].dropna().copy()

train_df_lstm = lstm_df[lstm_df["datetime"] < "2024-01-01"].copy()
val_df_lstm = lstm_df[(lstm_df["datetime"] >= "2024-01-01") & (lstm_df["datetime"] < "2025-01-01")].copy()
test_df_lstm = lstm_df[(lstm_df["datetime"] >= "2025-01-01") & (lstm_df["datetime"] < "2026-01-01")].copy()
test_2026_lstm = lstm_df[lstm_df["datetime"] >= "2026-01-01"].copy()

display(pd.DataFrame([
    {"Split": "Train", "Start": train_df_lstm["datetime"].min(), "End": train_df_lstm["datetime"].max(), "Rows": len(train_df_lstm)},
    {"Split": "Validation_2024", "Start": val_df_lstm["datetime"].min(), "End": val_df_lstm["datetime"].max(), "Rows": len(val_df_lstm)},
    {"Split": "Test_2025", "Start": test_df_lstm["datetime"].min(), "End": test_df_lstm["datetime"].max(), "Rows": len(test_df_lstm)},
    {"Split": "Test_2026_YTD", "Start": test_2026_lstm["datetime"].min(), "End": test_2026_lstm["datetime"].max(), "Rows": len(test_2026_lstm)},
]))

print(lstm_features)

In [ ]:
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_sequences(X, y, datetimes, seq_len):
    X_seq, y_seq, dt_seq = [], [], []

    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
        dt_seq.append(datetimes[i])

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32),
        pd.to_datetime(dt_seq),
    )

def prepare_lstm_bundle(train_df, val_df, test_df, test_2026_df, feature_names, target_name, seq_len, batch_size):
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    X_train_raw = train_df[feature_names].values
    X_val_raw = val_df[feature_names].values
    X_test_raw = test_df[feature_names].values
    X_test_2026_raw = test_2026_df[feature_names].values

    y_train_raw = train_df[[target_name]].values
    y_val_raw = val_df[[target_name]].values
    y_test_raw = test_df[[target_name]].values
    y_test_2026_raw = test_2026_df[[target_name]].values

    X_train_scaled = feature_scaler.fit_transform(X_train_raw)
    X_val_scaled = feature_scaler.transform(X_val_raw)
    X_test_scaled = feature_scaler.transform(X_test_raw)
    X_test_2026_scaled = feature_scaler.transform(X_test_2026_raw)

    y_train_scaled = target_scaler.fit_transform(y_train_raw)
    y_val_scaled = target_scaler.transform(y_val_raw)
    y_test_scaled = target_scaler.transform(y_test_raw)
    y_test_2026_scaled = target_scaler.transform(y_test_2026_raw)

    X_train_seq, y_train_seq, dt_train_seq = create_sequences(X_train_scaled, y_train_scaled, train_df["datetime"].values, seq_len)
    X_val_seq, y_val_seq, dt_val_seq = create_sequences(X_val_scaled, y_val_scaled, val_df["datetime"].values, seq_len)
    X_test_seq, y_test_seq, dt_test_seq = create_sequences(X_test_scaled, y_test_scaled, test_df["datetime"].values, seq_len)
    X_test_2026_seq, y_test_2026_seq, dt_test_2026_seq = create_sequences(
        X_test_2026_scaled, y_test_2026_scaled, test_2026_df["datetime"].values, seq_len
    )

    train_loader = DataLoader(SequenceDataset(X_train_seq, y_train_seq), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(SequenceDataset(X_val_seq, y_val_seq), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(SequenceDataset(X_test_seq, y_test_seq), batch_size=batch_size, shuffle=False)
    test_2026_loader = DataLoader(SequenceDataset(X_test_2026_seq, y_test_2026_seq), batch_size=batch_size, shuffle=False)

    return {
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "X_train_seq": X_train_seq,
        "y_train_seq": y_train_seq,
        "X_val_seq": X_val_seq,
        "y_val_seq": y_val_seq,
        "X_test_seq": X_test_seq,
        "y_test_seq": y_test_seq,
        "X_test_2026_seq": X_test_2026_seq,
        "y_test_2026_seq": y_test_2026_seq,
        "dt_train_seq": dt_train_seq,
        "dt_val_seq": dt_val_seq,
        "dt_test_seq": dt_test_seq,
        "dt_test_2026_seq": dt_test_2026_seq,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "test_2026_loader": test_2026_loader,
    }

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1, dropout=0.2):
        super().__init__()

        recurrent_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)

def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)

def predict_lstm(model, loader, device):
    model.eval()
    preds = []

    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            batch_preds = model(X_batch).cpu().numpy()
            preds.append(batch_preds)

    return np.vstack(preds)

def train_lstm(config, bundle, device, verbose=False):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = LSTMRegressor(
        input_size=bundle["X_train_seq"].shape[2],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    epochs = config["epochs"]
    patience = config["patience"]

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, bundle["train_loader"], criterion, optimizer, device)
        val_loss = evaluate_one_epoch(model, bundle["val_loader"], criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if verbose:
            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "history": history,
        "best_val_loss": best_val_loss,
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

lstm_param_grid = [
    {
        "seq_len": 24,
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "epochs": 60,
        "patience": 6,
    },
    {
        "seq_len": 72,
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "epochs": 60,
        "patience": 6,
    },
    {
        "seq_len": 72,
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "epochs": 60,
        "patience": 6,
    },
    {
        "seq_len": 168,
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "epochs": 60,
        "patience": 6,
    },
]

best_lstm_run = None
best_lstm_config = None
best_lstm_score = -np.inf

for config in lstm_param_grid:
    print("Running LSTM config:", config)

    bundle = prepare_lstm_bundle(
        train_df=train_df_lstm,
        val_df=val_df_lstm,
        test_df=test_df_lstm,
        test_2026_df=test_2026_lstm,
        feature_names=lstm_features,
        target_name=target,
        seq_len=config["seq_len"],
        batch_size=config["batch_size"],
    )

    run = train_lstm(config, bundle, device, verbose=False)

    val_pred_scaled = predict_lstm(run["model"], bundle["val_loader"], device)
    val_pred = bundle["target_scaler"].inverse_transform(val_pred_scaled).flatten()
    y_val_actual = bundle["target_scaler"].inverse_transform(bundle["y_val_seq"]).flatten()

    row = record_metrics(search_results, "LSTM", config, "Validation_2024", y_val_actual, val_pred)
    search_results[-1]["Epochs_Trained"] = len(run["history"]["train_loss"])
    print_row(row)

    if row["R_squared"] > best_lstm_score:
        best_lstm_score = row["R_squared"]
        best_lstm_config = config
        best_lstm_run = {
            "config": config,
            "bundle": bundle,
            "model": run["model"],
            "history": run["history"],
        }

lstm_search_df = (
    pd.DataFrame([r for r in search_results if r["Model"] == "LSTM"])
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

display(lstm_search_df)
print("Best LSTM config:", best_lstm_config)
print("Best LSTM Validation R_squared:", best_lstm_score)

In [ ]:
# Best LSTM: validation + 2025 test + 2026 YTD
bundle = best_lstm_run["bundle"]
best_lstm_model = best_lstm_run["model"]

for split_name, loader_key, y_key, dt_key in [
    ("Validation_2024", "val_loader", "y_val_seq", "dt_val_seq"),
    ("Test_2025", "test_loader", "y_test_seq", "dt_test_seq"),
    ("Test_2026_YTD", "test_2026_loader", "y_test_2026_seq", "dt_test_2026_seq"),
]:
    pred_scaled = predict_lstm(best_lstm_model, bundle[loader_key], device)
    pred = bundle["target_scaler"].inverse_transform(pred_scaled).flatten()
    y_actual = bundle["target_scaler"].inverse_transform(bundle[y_key]).flatten()

    row = record_metrics(final_results, "LSTM", best_lstm_config, split_name, y_actual, pred)
    print_row(row)
    store_predictions("LSTM", split_name, bundle[dt_key], y_actual, pred)

plt.figure(figsize=(10, 4))
plt.plot(best_lstm_run["history"]["train_loss"], label="Train Loss")
plt.plot(best_lstm_run["history"]["val_loss"], label="Validation Loss")
plt.title("Best LSTM Training Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
search_results_df = pd.DataFrame(search_results).sort_values(
    ["Model", "R_squared"], ascending=[True, False]
).reset_index(drop=True)

final_results_df = pd.DataFrame(final_results).sort_values(
    ["Split", "R_squared"], ascending=[True, False]
).reset_index(drop=True)

best_validation_by_model = (
    final_results_df[final_results_df["Split"] == "Validation_2024"]
    .sort_values("R_squared", ascending=False)
    .groupby("Model", as_index=False)
    .first()
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

search_results_path = OUTPUT_DIR / "model_search_results.csv"
final_results_path = OUTPUT_DIR / "model_final_results.csv"

search_results_df.to_csv(search_results_path, index=False)
final_results_df.to_csv(final_results_path, index=False)

display(search_results_df)
display(final_results_df)
display(best_validation_by_model)

print("Saved:")
print(search_results_path.resolve())
print(final_results_path.resolve())

In [ ]:
def build_plot_frame(split_name):
    pretty_names = {
        "Baseline_CurrentERCOT": "Baseline",
        "RandomForest": "Random Forest",
        "LightGBM": "LightGBM",
        "LSTM": "LSTM",
    }

    keys = [k for k in prediction_frames.keys() if k[1] == split_name]
    if not keys:
        return None

    plot_df = None
    for model_name, _ in keys:
        frame = prediction_frames[(model_name, split_name)].copy()
        label = pretty_names.get(model_name, model_name)
        frame = frame.rename(columns={"pred": label})

        if plot_df is None:
            plot_df = frame[["datetime", "actual", label]].copy()
        else:
            plot_df = plot_df.merge(frame[["datetime", label]], on="datetime", how="inner")

    return plot_df.sort_values("datetime").reset_index(drop=True)

def plot_predictions(split_name, n_hours=24 * 14, start_idx=0):
    plot_df = build_plot_frame(split_name)
    if plot_df is None or len(plot_df) == 0:
        print(f"No prediction frame available for {split_name}")
        return

    sample = plot_df.iloc[start_idx:start_idx + n_hours]

    plt.figure(figsize=(15, 5))
    plt.plot(sample["datetime"], sample["actual"], label="Actual", linewidth=2)

    for col in sample.columns:
        if col not in ["datetime", "actual"]:
            plt.plot(sample["datetime"], sample[col], label=col)

    plt.title(f"Model Comparison - {split_name}")
    plt.xlabel("Datetime")
    plt.ylabel("Load")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_predictions("Test_2025")
plot_predictions("Test_2026_YTD")

In [ ]:
def plot_event_window(split_name, start_date, end_date):
    plot_df = build_plot_frame(split_name)
    if plot_df is None or len(plot_df) == 0:
        print(f"No prediction frame available for {split_name}")
        return

    mask = (plot_df["datetime"] >= pd.to_datetime(start_date)) & (plot_df["datetime"] <= pd.to_datetime(end_date))
    sample = plot_df.loc[mask].copy()

    if sample.empty:
        print("No rows found in that date range.")
        return

    plt.figure(figsize=(15, 5))
    plt.plot(sample["datetime"], sample["actual"], label="Actual", linewidth=2)

    for col in sample.columns:
        if col not in ["datetime", "actual"]:
            plt.plot(sample["datetime"], sample[col], label=col)

    plt.title(f"Model Comparison During Event Window: {start_date} to {end_date}")
    plt.xlabel("Datetime")
    plt.ylabel("Load")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_event_window("Test_2025", "2025-01-17", "2025-01-23")

In [ ]:
def metrics_for_window(split_name, start_date, end_date):
    rows = []

    for (model_name, stored_split), frame in prediction_frames.items():
        if stored_split != split_name:
            continue

        mask = (frame["datetime"] >= pd.to_datetime(start_date)) & (frame["datetime"] <= pd.to_datetime(end_date))
        sample = frame.loc[mask].copy()

        if len(sample) == 0:
            continue

        metrics = compute_metrics(sample["actual"], sample["pred"])
        rows.append({
            "Model": model_name,
            "Split": split_name,
            "Window_Start": start_date,
            "Window_End": end_date,
            **metrics,
        })

    return pd.DataFrame(rows).sort_values("R_squared", ascending=False).reset_index(drop=True)

event_metrics_df = metrics_for_window("Test_2025", "2025-01-17", "2025-01-23")
display(event_metrics_df)
event_metrics_df.to_csv(OUTPUT_DIR / "event_metrics_2025_01_17_to_2025_01_23.csv", index=False)

In [ ]:
summary_2025 = (
    final_results_df[final_results_df["Split"] == "Test_2025"]
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

summary_2026 = (
    final_results_df[final_results_df["Split"] == "Test_2026_YTD"]
    .sort_values("R_squared", ascending=False)
    .reset_index(drop=True)
)

print("Best models on 2025:")
display(summary_2025[["Model", "MAE", "RMSE", "MAPE", "R_squared", "Config"]])

print("Best models on 2026 YTD:")
display(summary_2026[["Model", "MAE", "RMSE", "MAPE", "R_squared", "Config"]])